In [ ]:
from collections import deque, defaultdict

def bfs_task_scheduling(tasks, dependencies):
    """
    tasks: list of all task names
    dependencies: list of tuples (A, B) meaning "A must happen before B"
    """
    
    # Step 1: Build the graph and count incoming edges (in-degree)
    graph = defaultdict(list)
    in_degree = {task: 0 for task in tasks}
    
    for before, after in dependencies:
        graph[before].append(after)
        in_degree[after] += 1
    
    # Step 2: Start with tasks that have NO prerequisites (in-degree 0)
    queue = deque([task for task in tasks if in_degree[task] == 0])
    
    schedule = []
    
    # Step 3: Process the queue
    while queue:
        current = queue.popleft()
        schedule.append(current)
        
        # "Remove" current's edges by decrementing in-degree of its neighbors
        for neighbor in graph[current]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    # Step 4: Check for a cycle (impossible schedule)
    if len(schedule) != len(tasks):
        return None  # There's a circular dependency - no valid order exists
    
    return schedule


# --- Example Usage ---
tasks = ["wake_up", "shower", "brush_teeth", "eat_breakfast", "go_to_work"]

dependencies = [
    ("wake_up", "shower"),
    ("wake_up", "brush_teeth"),
    ("brush_teeth", "eat_breakfast"),
    ("shower", "go_to_work"),
    ("eat_breakfast", "go_to_work"),
]

result = bfs_task_scheduling(tasks, dependencies)
print(result)

In [1]:
import heapq
from dataclasses import dataclass, field

@dataclass
class Job:
    name: str
    urgency: int      # lower number = more urgent
    duration: int      # how long the job takes (minutes)

class TaskScheduler:
    def __init__(self):
        self.heap = []
        self._counter = 0   # tie-breaker for equal urgency

    def add_job(self, job: Job):
        # Tuple: (urgency, insertion_order, job)
        heapq.heappush(self.heap, (job.urgency, self._counter, job))
        self._counter += 1
        print(f"Added: {job.name} (urgency={job.urgency})")

    def run_next(self):
        if not self.heap:
            print("No jobs left to run.")
            return None
        
        urgency, order, job = heapq.heappop(self.heap)
        print(f"Running: {job.name} (urgency={job.urgency}, duration={job.duration}min)")
        return job

    def run_all(self):
        print("\n--- Running all jobs in priority order ---")
        while self.heap:
            self.run_next()

    def is_empty(self):
        return len(self.heap) == 0

    def pending_count(self):
        return len(self.heap)


# --- Example Usage ---
scheduler = TaskScheduler()

scheduler.add_job(Job(name="Send weekly report", urgency=5, duration=10))
scheduler.add_job(Job(name="Fix production outage", urgency=1, duration=30))
scheduler.add_job(Job(name="Backup database", urgency=3, duration=15))
scheduler.add_job(Job(name="Update dependencies", urgency=4, duration=5))
scheduler.add_job(Job(name="Restart crashed server", urgency=1, duration=2))

scheduler.run_all()

Added: Send weekly report (urgency=5)
Added: Fix production outage (urgency=1)
Added: Backup database (urgency=3)
Added: Update dependencies (urgency=4)
Added: Restart crashed server (urgency=1)

--- Running all jobs in priority order ---
Running: Fix production outage (urgency=1, duration=30min)
Running: Restart crashed server (urgency=1, duration=2min)
Running: Backup database (urgency=3, duration=15min)
Running: Update dependencies (urgency=4, duration=5min)
Running: Send weekly report (urgency=5, duration=10min)
